# 03 - Build Cohort

Mục tiêu:
1. Đọc `project_inventory.csv` đã tạo ở notebook 02.
2. Chọn cohort project theo tiêu chí khai báo trong `configs/experiment.yaml`.
3. Extract đúng các project đã chọn từ MongoDB, không tải toàn bộ 2,7 triệu issue vào RAM.
4. Tạo survival target:
   - `event=True`: đã resolve;
   - `event=False`: right-censored;
   - `duration_days`: thời gian từ creation đến resolution/censoring.
5. Lưu `data/processed/cohort.parquet`.

**Quan trọng:** kiểm tra lại `schema:` trong `experiment.yaml` bằng kết quả của notebook 01 trước khi chạy extraction.

In [1]:
from pathlib import Path
import sys, yaml, pandas as pd, numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

with open(ROOT / "configs" / "experiment.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

print("ROOT =", ROOT)
print("random_seed =", CFG["random_seed"])

ROOT = d:\VNUK\Eureka 2026\Eureka_2026
random_seed = 42


In [2]:
from pymongo import MongoClient

client = MongoClient(CFG["mongodb"]["uri"], serverSelectionTimeoutMS=5000)
print("Ping:", client.admin.command("ping"))

user_dbs = [
    d for d in client.list_database_names()
    if d not in {"admin", "config", "local"}
]

db_name = CFG["mongodb"].get("database", "AUTO")
if not db_name or str(db_name).upper() == "AUTO":
    if len(user_dbs) != 1:
        raise RuntimeError(
            f"MongoDB có {len(user_dbs)} user databases: {user_dbs}. "
            "Hãy ghi đúng tên DB vào configs/experiment.yaml."
        )
    db_name = user_dbs[0]

db = client[db_name]
print("Selected DB:", db_name)
print("Collections:", len(db.list_collection_names()))

Ping: {'ok': 1.0}
Selected DB: JiraReposAnon
Collections: 16


In [3]:
inventory_path = ROOT / "results" / "tables" / "project_inventory.csv"
if not inventory_path.exists():
    raise FileNotFoundError(
        f"Không thấy {inventory_path}. Notebook 02 phải lưu project_inventory.csv trước."
    )

projects_df = pd.read_csv(inventory_path)
required = {"repository", "project", "issues", "resolved", "unresolved"}
missing = required - set(projects_df.columns)
if missing:
    raise ValueError(f"project_inventory.csv thiếu cột: {missing}")

if "censoring_rate" not in projects_df:
    projects_df["censoring_rate"] = projects_df["unresolved"] / projects_df["issues"]

projects_df.head()

,repository,project,issues,resolved,unresolved,resolved_rate,censoring_rate
0,Apache,TOREE,528,411,117,0.778409,0.221591
1,Apache,XALANJ,954,697,257,0.730608,0.269392
2,Apache,VELOCITYSB,9,9,0,1.000000,0.000000
3,Apache,JSIEVE,115,100,15,0.869565,0.130435
4,Apache,ACE,539,492,47,0.912801,0.087199


In [4]:
sel = CFG["selection"]

eligible = projects_df[
    (projects_df["issues"] >= sel["min_issues"])
    & (projects_df["resolved"] >= sel["min_resolved"])
    & (projects_df["unresolved"] >= sel["min_unresolved"])
    & (projects_df["censoring_rate"] >= sel["min_censoring_rate"])
    & (projects_df["censoring_rate"] <= sel["max_censoring_rate"])
].copy()

# Score chỉ dùng để chọn cohort vận hành cân bằng event/censoring,
# KHÔNG phải biến dự báo.
eligible["survival_info_score"] = np.sqrt(
    (eligible["resolved"] + 1) * (eligible["unresolved"] + 1)
)

eligible = eligible.sort_values(
    ["survival_info_score", "issues"],
    ascending=False,
)

print("Eligible projects:", len(eligible))
eligible.head(30)

Eligible projects: 325


,repository,project,issues,resolved,unresolved,resolved_rate,censoring_rate,survival_info_score
844,Mojang,MC,213845,205547,8298,0.961196,0.038804,41301.850467
884,Qt,QTBUG,97172,77520,19652,0.797761,0.202239,39032.297050
841,Mojang,MCPE,144778,140317,4461,0.969187,0.030813,25021.968668
690,Jira,JRASERVER,47225,38678,8547,0.819015,0.180985,18183.181570
867,MongoDB,SERVER,58928,52972,5956,0.898928,0.101072,17764.013088
710,Jira,CONFSERVER,43910,37290,6620,0.849237,0.150763,15713.169986
1192,Sonatype,OSSRH,74960,72193,2767,0.963087,0.036913,14136.229766
1180,Sakai,SAK,43351,39086,4265,0.901617,0.098383,12912.983466
517,Apache,HIVE,25731,18516,7215,0.719599,0.280401,11559.354307
696,Jira,JRACLOUD,27916,22142,5774,0.793165,0.206835,11308.219356


In [5]:
# Chọn tối đa N project, đồng thời tránh lấy quá nhiều project từ cùng 1 repository.
max_projects = int(sel["max_projects"])
cap = int(sel["max_projects_per_repository"])

chosen = []
repo_count = {}

for row in eligible.itertuples(index=False):
    repo = str(row.repository)
    if repo_count.get(repo, 0) >= cap:
        continue
    chosen.append(row)
    repo_count[repo] = repo_count.get(repo, 0) + 1
    if len(chosen) >= max_projects:
        break

selected = pd.DataFrame(chosen)

if selected.empty:
    raise RuntimeError(
        "Không có project nào đạt filter. Hãy xem distribution ở notebook 02 "
        "và điều chỉnh selection thresholds trong experiment.yaml."
    )

selected_path = ROOT / "results" / "tables" / "selected_projects.csv"
selected_path.parent.mkdir(parents=True, exist_ok=True)
selected.to_csv(selected_path, index=False)

print("Selected projects:", len(selected))
selected[["repository", "project", "issues", "resolved", "unresolved", "censoring_rate"]]

Selected projects: 12


,repository,project,issues,resolved,unresolved,censoring_rate
0,Mojang,MC,213845,205547,8298,0.038804
1,Qt,QTBUG,97172,77520,19652,0.202239
2,Mojang,MCPE,144778,140317,4461,0.030813
3,Jira,JRASERVER,47225,38678,8547,0.180985
4,MongoDB,SERVER,58928,52972,5956,0.101072
5,Jira,CONFSERVER,43910,37290,6620,0.150763
6,Sonatype,OSSRH,74960,72193,2767,0.036913
7,Sakai,SAK,43351,39086,4265,0.098383
8,Apache,HIVE,25731,18516,7215,0.280401
9,Jira,JRACLOUD,27916,22142,5774,0.206835


## Extract issue-level data

Notebook này stream từng project thành từng file Parquet trong `data/interim/issues_flat/`.

Nếu extraction bị dừng giữa chừng, chạy lại sẽ **skip các file đã tồn tại**.

In [6]:
from src.extract import extract_selected_projects

schema = CFG["schema"]
landmark_days = int(CFG["features"]["landmark_days"])

manifest = extract_selected_projects(
    db=db,
    selected_projects=selected,
    schema=schema,
    out_dir=ROOT / "data" / "interim" / "issues_flat",
    landmark_days=landmark_days,
    overwrite=False,
)

manifest_path = ROOT / "results" / "tables" / "extraction_manifest.csv"
manifest.to_csv(manifest_path, index=False)
manifest

Extracting Mojang / MC ...
  -> written 213845
Extracting Qt / QTBUG ...
  -> written 97172
Extracting Mojang / MCPE ...
  -> written 144778
Extracting Jira / JRASERVER ...
  -> written 47225
Extracting MongoDB / SERVER ...
  -> written 58928
Extracting Jira / CONFSERVER ...
  -> written 43910
Extracting Sonatype / OSSRH ...
  -> written 74960
Extracting Sakai / SAK ...
  -> written 43351
Extracting Apache / HIVE ...
  -> written 25731
Extracting Jira / JRACLOUD ...
  -> written 27916
Extracting Apache / FLEX ...
  -> written 35390
Extracting MariaDB / MDEV ...
  -> written 22437


,repository,project,rows,file,status
0,Mojang,MC,213845,d:\VNUK\Eureka 2026\Eureka_2026\data\interim\i...,written
1,Qt,QTBUG,97172,d:\VNUK\Eureka 2026\Eureka_2026\data\interim\i...,written
2,Mojang,MCPE,144778,d:\VNUK\Eureka 2026\Eureka_2026\data\interim\i...,written
3,Jira,JRASERVER,47225,d:\VNUK\Eureka 2026\Eureka_2026\data\interim\i...,written
4,MongoDB,SERVER,58928,d:\VNUK\Eureka 2026\Eureka_2026\data\interim\i...,written
5,Jira,CONFSERVER,43910,d:\VNUK\Eureka 2026\Eureka_2026\data\interim\i...,written
6,Sonatype,OSSRH,74960,d:\VNUK\Eureka 2026\Eureka_2026\data\interim\i...,written
7,Sakai,SAK,43351,d:\VNUK\Eureka 2026\Eureka_2026\data\interim\i...,written
8,Apache,HIVE,25731,d:\VNUK\Eureka 2026\Eureka_2026\data\interim\i...,written
9,Jira,JRACLOUD,27916,d:\VNUK\Eureka 2026\Eureka_2026\data\interim\i...,written


In [7]:
from src.cohort import load_flat_dataset, build_survival_cohort

flat = load_flat_dataset(ROOT / "data" / "interim" / "issues_flat")
print("Flat rows:", len(flat))
print("Projects:", flat["project"].nunique())

cohort, quality_report = build_survival_cohort(flat)

cohort_path = ROOT / "data" / "processed" / "cohort.parquet"
cohort_path.parent.mkdir(parents=True, exist_ok=True)
cohort.to_parquet(cohort_path, index=False)

quality_path = ROOT / "results" / "tables" / "cohort_quality.csv"
quality_report.to_csv(quality_path, index=False)

quality_report

Flat rows: 835643
Projects: 12


,metric,value
0,input_rows,835643.000000
1,rows_without_created_removed,0.000000
2,invalid_duration_removed,0.000000
3,zero_duration_adjusted,21.000000
4,final_rows,835643.000000
5,events_resolved,751889.000000
6,censored_unresolved,83754.000000
7,censoring_rate,0.100227


In [8]:
# Sanity checks
assert (cohort["duration_days"] > 0).all()
assert cohort["event"].dtype == bool
assert cohort["created"].notna().all()

summary = (
    cohort.groupby(["repository", "project"])
    .agg(
        issues=("issue_key", "size"),
        events=("event", "sum"),
        median_duration_days=("duration_days", "median"),
    )
    .reset_index()
)
summary["censored"] = summary["issues"] - summary["events"]
summary["censoring_rate"] = summary["censored"] / summary["issues"]

summary.to_csv(
    ROOT / "results" / "tables" / "cohort_project_summary.csv",
    index=False,
)

summary

,repository,project,issues,events,median_duration_days,censored,censoring_rate
0,Apache,FLEX,35390,31564,132.300249,3826,0.108110
1,Apache,HIVE,25731,18516,26.688252,7215,0.280401
2,Jira,CONFSERVER,43910,37290,443.209896,6620,0.150763
3,Jira,JRACLOUD,27916,22142,805.739039,5774,0.206835
4,Jira,JRASERVER,47225,38678,426.490706,8547,0.180985
5,MariaDB,MDEV,22437,16064,103.308356,6373,0.284040
6,Mojang,MC,213845,205547,0.177396,8298,0.038804
7,Mojang,MCPE,144778,140317,1.122054,4461,0.030813
8,MongoDB,SERVER,58928,52972,22.137760,5956,0.101072
9,Qt,QTBUG,97172,77520,110.856227,19652,0.202239
